# Data Mining Assignment — Apache Spark Data Lake Preprocessing

**Dataset:** `sales.csv`  
**Environment:** Google Colab + PySpark  

This notebook creates a simple two-layer Data Lake using Apache Spark.

```text
data_lake/
├── bronze/
│   └── sales.csv
└── silver/
    └── preprocessed_sales/

In [1]:
!pip install -q pyspark

In [2]:
import os
import shutil
from google.colab import files

bronze_dir = "data_lake/bronze"
silver_dir = "data_lake/silver/preprocessed_sales"
silver_root = "data_lake/silver"

os.makedirs(bronze_dir, exist_ok=True)
os.makedirs(silver_dir, exist_ok=True)

print("Upload the provided sales.csv file:")
uploaded = files.upload()

if "sales.csv" not in uploaded:
    raise FileNotFoundError("Please upload a file named sales.csv")

shutil.copyfile("sales.csv", os.path.join(bronze_dir, "sales.csv"))

print("\nData Lake structure created:")
for root, dirs, files_list in os.walk("data_lake"):
    level = root.replace("data_lake", "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    for name in sorted(files_list):
        print(f"{indent}    {name}")

Upload the provided sales.csv file:


Saving sales.csv to sales.csv

Data Lake structure created:
data_lake/
    .ipynb_checkpoints/
    bronze/
        sales.csv
    silver/
        .ipynb_checkpoints/
        preprocessed_sales/
            ._SUCCESS.crc
            .part-00000-d75a373b-847e-407f-8260-7d6f69201f4c-c000.csv.crc
            .ipynb_checkpoints/


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("SalesDataLake")
    .master("local[*]")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

Spark version: 4.0.4


In [4]:
bronze_path = "data_lake/bronze/sales.csv"

bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE")
    .csv(bronze_path)
)

print("=== Schema ===")
bronze_df.printSchema()

print("\n=== Sample Records ===")
bronze_df.show(5, truncate=False)

bronze_count = bronze_df.count()
print("\nTotal records:", bronze_count)
print("Total columns:", len(bronze_df.columns))

=== Schema ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)


=== Sample Records ===
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+-------

In [5]:
unique_count = bronze_df.dropDuplicates().count()
duplicate_rows = bronze_count - unique_count

print("Exact duplicate rows:", duplicate_rows)

print("\nDuplicate records:")
(
    bronze_df
    .groupBy(bronze_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .orderBy("order_id")
    .show(10, truncate=False)
)

Exact duplicate rows: 10

Duplicate records:
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100055  |C0270      |Pranav Iyer     |Dry Fruits Pack  |Grocery        |7       |930       |12              |Cash on Delivery|Coimbatore|Tamil Nadu    |2026-08-20|Delivered   |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C0149      |Manish Pillai

In [6]:
missing_report = bronze_df.select([
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in bronze_df.columns
])

print("=== Missing / Blank Values ===")
missing_report.show(truncate=False)

=== Missing / Blank Values ===
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [7]:
text_columns = [
    "customer_id", "customer_name", "product", "category",
    "payment_method", "city", "state", "order_status"
]

print("=== Extra Space Issues ===")

space_check = bronze_df.select([
    F.sum(
        F.when(
            F.col(c).isNotNull() &
            (
                (F.col(c) != F.trim(F.col(c))) |
                F.col(c).rlike(r"\s{2,}")
            ),
            1
        ).otherwise(0)
    ).alias(c)
    for c in text_columns
])

space_check.show(truncate=False)

print("\n=== Category Values ===")
bronze_df.groupBy("category").count().orderBy("category").show(truncate=False)

print("\n=== City Values ===")
bronze_df.groupBy("city").count().orderBy("city").show(truncate=False)

=== Extra Space Issues ===
+-----------+-------------+-------+--------+--------------+----+-----+------------+
|customer_id|customer_name|product|category|payment_method|city|state|order_status|
+-----------+-------------+-------+--------+--------------+----+-----+------------+
|0          |3            |0      |0       |0             |3   |0    |0           |
+-----------+-------------+-------+--------+--------------+----+-----+------------+


=== Category Values ===
+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |136  |
|Books          |128  |
|CLOTHING       |1    |
|Clothing       |108  |
|ELECTRONICS    |1    |
|Electronics    |103  |
|Furniture      |132  |
|Grocery        |136  |
|Home Appliances|115  |
|SPORTS         |1    |
|Sports         |136  |
|beauty         |2    |
|sports         |1    |
+---------------+-----+


=== City Values ===
+-----------+-----+
|city       |count|
+-----------+-----+
|NULL       |8    |
| Hyderabad |2   

In [8]:
q_raw = F.trim(F.col("quantity"))
p_raw = F.trim(F.col("unit_price"))
d_raw = F.trim(F.col("discount_percent"))

q_val = q_raw.cast("int")
p_val = p_raw.cast("double")
d_val = d_raw.cast("double")

quantity_missing = q_raw.isNull() | (q_raw == "")
price_missing = p_raw.isNull() | (p_raw == "")
discount_missing = d_raw.isNull() | (d_raw == "")

numeric_report = bronze_df.select(
    F.sum(
        F.when(
            (~quantity_missing) &
            (q_val.isNull() | (q_val <= 0)),
            1
        ).otherwise(0)
    ).alias("invalid_quantity"),

    F.sum(
        F.when(
            (~price_missing) &
            (p_val.isNull() | (p_val <= 0)),
            1
        ).otherwise(0)
    ).alias("invalid_unit_price"),

    F.sum(
        F.when(
            (~discount_missing) &
            (
                d_val.isNull() |
                (d_val < 0) |
                (d_val > 100)
            ),
            1
        ).otherwise(0)
    ).alias("invalid_discount")
)

print("=== Invalid Numerical Values ===")
numeric_report.show()

=== Invalid Numerical Values ===
+----------------+------------------+----------------+
|invalid_quantity|invalid_unit_price|invalid_discount|
+----------------+------------------+----------------+
|              12|                 8|               8|
+----------------+------------------+----------------+



In [9]:
date_raw = F.trim(F.col("order_date"))

standard_date = F.to_date(date_raw, "yyyy-MM-dd")

parsed_date = F.coalesce(
    standard_date,
    F.to_date(date_raw, "yyyy/MM/dd"),
    F.to_date(date_raw, "dd/MM/yyyy"),
    F.to_date(date_raw, "MM-dd-yyyy")
)

date_report = bronze_df.select(
    F.sum(
        F.when(
            date_raw.isNull() | (date_raw == ""),
            1
        ).otherwise(0)
    ).alias("missing_dates"),

    F.sum(
        F.when(
            date_raw.isNotNull() &
            (date_raw != "") &
            standard_date.isNull() &
            parsed_date.isNotNull(),
            1
        ).otherwise(0)
    ).alias("valid_nonstandard_dates"),

    F.sum(
        F.when(
            date_raw.isNotNull() &
            (date_raw != "") &
            parsed_date.isNull(),
            1
        ).otherwise(0)
    ).alias("invalid_dates")
)

print("=== Date Check ===")
date_report.show()

print("\n=== Invalid Date Records ===")
(
    bronze_df
    .withColumn("parsed_date", parsed_date)
    .filter(
        date_raw.isNotNull() &
        (date_raw != "") &
        F.col("parsed_date").isNull()
    )
    .select("order_id", "order_date")
    .show(10, truncate=False)
)

=== Date Check ===
+-------------+-----------------------+-------------+
|missing_dates|valid_nonstandard_dates|invalid_dates|
+-------------+-----------------------+-------------+
|            0|                      2|            4|
+-------------+-----------------------+-------------+


=== Invalid Date Records ===
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
+--------+----------+



In [10]:
clean_df = bronze_df.dropDuplicates()

after_dedup = clean_df.count()

print("Rows before preprocessing:", bronze_count)
print("Rows after exact deduplication:", after_dedup)
print("Rows removed:", bronze_count - after_dedup)

Rows before preprocessing: 1000
Rows after exact deduplication: 990
Rows removed: 10


In [11]:
def tidy_text(column_name):
    cleaned = F.trim(
        F.regexp_replace(F.col(column_name), r"\s+", " ")
    )
    return F.when(cleaned == "", None).otherwise(cleaned)

for column_name in text_columns:
    clean_df = clean_df.withColumn(
        column_name,
        tidy_text(column_name)
    )

# IDs are conventionally uppercase
clean_df = clean_df.withColumn(
    "customer_id",
    F.upper(F.col("customer_id"))
)

# These fields benefit from consistent title-style capitalization
for column_name in ["customer_name", "category", "city", "state", "order_status"]:
    clean_df = clean_df.withColumn(
        column_name,
        F.initcap(F.col(column_name))
    )

clean_df.select(text_columns).show(5, truncate=False)

+-----------+--------------+----------------------+-----------+----------------+---------+-------------+------------+
|customer_id|customer_name |product               |category   |payment_method  |city     |state        |order_status|
+-----------+--------------+----------------------+-----------+----------------+---------+-------------+------------+
|C0006      |Aditi Tripathi|Football              |Sports     |Cash on Delivery|Jaipur   |Rajasthan    |Delivered   |
|C0140      |Sneha Sharma  |Competitive Exam Guide|Books      |Cash on Delivery|Pune     |Maharashtra  |Delivered   |
|C0265      |Pranav Joshi  |Bedside Table         |Furniture  |Debit Card      |Hyderabad|Telangana    |Processing  |
|C0070      |Sneha Chopra  |Face Wash             |Beauty     |Debit Card      |Lucknow  |Uttar Pradesh|Shipped     |
|C0090      |Abhinav Pillai|Mechanical Keyboard   |Electronics|Debit Card      |Kolkata  |West Bengal  |Delivered   |
+-----------+--------------+----------------------+-----

In [12]:
# Text fields
for column_name in ["customer_name", "payment_method", "city", "category", "state", "order_status"]:
    clean_df = clean_df.fillna("Unknown", subset=[column_name])

# Keep track of genuinely missing numeric values
clean_df = (
    clean_df
    .withColumn(
        "_quantity_missing",
        F.col("quantity").isNull() | (F.trim(F.col("quantity")) == "")
    )
    .withColumn(
        "_price_missing",
        F.col("unit_price").isNull() | (F.trim(F.col("unit_price")) == "")
    )
    .withColumn(
        "_discount_missing",
        F.col("discount_percent").isNull() | (F.trim(F.col("discount_percent")) == "")
    )
)

# Cast numeric columns
clean_df = (
    clean_df
    .withColumn("_quantity_value", F.col("quantity").cast("int"))
    .withColumn("_price_value", F.col("unit_price").cast("double"))
    .withColumn("_discount_value", F.col("discount_percent").cast("double"))
)

# Validate domain ranges
clean_df = (
    clean_df
    .withColumn(
        "_quantity_valid",
        F.when(
            F.col("_quantity_value") > 0,
            F.col("_quantity_value")
        )
    )
    .withColumn(
        "_price_valid",
        F.when(
            F.col("_price_value") > 0,
            F.col("_price_value")
        )
    )
    .withColumn(
        "_discount_valid",
        F.when(
            F.col("_discount_value").between(0, 100),
            F.col("_discount_value")
        )
    )
)

# Median values are calculated only from valid data
quantity_median = clean_df.approxQuantile("_quantity_valid", [0.5], 0.01)[0]
price_median = clean_df.approxQuantile("_price_valid", [0.5], 0.01)[0]
discount_median = clean_df.approxQuantile("_discount_valid", [0.5], 0.01)[0]

clean_df = (
    clean_df
    .withColumn(
        "quantity",
        F.when(
            F.col("_quantity_missing"),
            F.lit(int(round(quantity_median)))
        ).otherwise(F.col("_quantity_valid"))
    )
    .withColumn(
        "unit_price",
        F.when(
            F.col("_price_missing"),
            F.lit(float(price_median))
        ).otherwise(F.col("_price_valid"))
    )
    .withColumn(
        "discount_percent",
        F.when(
            F.col("_discount_missing"),
            F.lit(float(discount_median))
        ).otherwise(F.col("_discount_valid"))
    )
)

In [13]:
clean_df = clean_df.withColumn(
    "_parsed_order_date",
    F.coalesce(
        F.to_date(F.trim(F.col("order_date")), "yyyy-MM-dd"),
        F.to_date(F.trim(F.col("order_date")), "yyyy/MM/dd"),
        F.to_date(F.trim(F.col("order_date")), "dd/MM/yyyy"),
        F.to_date(F.trim(F.col("order_date")), "MM-dd-yyyy")
    )
)

clean_df = clean_df.withColumn(
    "order_date",
    F.col("_parsed_order_date")
)

clean_df.select("order_id", "order_date").show(10, truncate=False)

+--------+----------+
|order_id|order_date|
+--------+----------+
|100124  |2026-06-29|
|100530  |2026-06-09|
|100576  |2026-03-23|
|100023  |2026-03-24|
|100798  |2026-04-09|
|100855  |2025-08-21|
|100904  |2026-04-14|
|100147  |2025-07-15|
|100254  |2025-02-17|
|100462  |2025-03-02|
+--------+----------+
only showing top 10 rows


In [14]:
temporary_columns = [
    "_quantity_missing",
    "_price_missing",
    "_discount_missing",
    "_quantity_value",
    "_price_value",
    "_discount_value",
    "_quantity_valid",
    "_price_valid",
    "_discount_valid",
    "_parsed_order_date"
]

clean_df = clean_df.drop(*temporary_columns)

clean_df = clean_df.select(
    "order_id",
    "customer_id",
    "customer_name",
    "product",
    "category",
    "quantity",
    "unit_price",
    "discount_percent",
    "payment_method",
    "city",
    "state",
    "order_date",
    "order_status"
)

clean_df.printSchema()
clean_df.show(10, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)

+--------+-----------+--------------+----------------------+-----------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name |product               |category   |quantity|unit_price|discount_percent|payment_method  |city     |state        |order_date|order_status|
+--------+-----------+--------------+----------------------+-----------+--------+----------+-----

In [15]:
silver_count = clean_df.count()

print("=== Final Verification ===")
print("Bronze row count :", bronze_count)
print("Silver row count :", silver_count)
print("Rows removed     :", bronze_count - silver_count)

print("\nFinal missing-value check:")
clean_df.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in clean_df.columns
]).show(truncate=False)

=== Final Verification ===
Bronze row count : 1000
Silver row count : 990
Rows removed     : 10

Final missing-value check:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |0            |0      |0       |12      |8         |8               |0             |0   |0    |4         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [16]:
import glob

spark_output = "data_lake/silver/preprocessed_sales"
submission_csv = "data_lake/silver/sales_silver.csv"

(
    clean_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("dateFormat", "yyyy-MM-dd")
    .csv(spark_output)
)

part_file = glob.glob(os.path.join(spark_output, "part-*.csv"))[0]

shutil.copyfile(part_file, submission_csv)

print("Silver Spark output:", spark_output)
print("Submission CSV      :", submission_csv)

Silver Spark output: data_lake/silver/preprocessed_sales
Submission CSV      : data_lake/silver/sales_silver.csv


In [17]:
silver_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(submission_csv)
)

print("=== Silver Data ===")
print("Rows:", silver_df.count())

print("\n=== Silver Schema ===")
silver_df.printSchema()

print("\n=== Silver Sample ===")
silver_df.show(10, truncate=False)

=== Silver Data ===
Rows: 990

=== Silver Schema ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)


=== Silver Sample ===
+--------+-----------+--------------+----------------------+-----------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name |product               |category   |quantity|unit_price|discount_percent|payment_method  |city     |state        |order_date|order_status|
+--------+-----------+----

## Preprocessing Summary

The following preprocessing operations were performed:

1. Preserved the original `sales.csv` in the Bronze layer.
2. Removed exact duplicate rows.
3. Trimmed leading/trailing spaces and collapsed repeated spaces in text fields.
4. Standardized obvious capitalization inconsistencies in categorical and location fields.
5. Replaced missing descriptive text values with `Unknown`.
6. Converted `quantity`, `unit_price`, and `discount_percent` to numeric values and applied basic range validation.
7. Used median values for genuinely missing numerical entries while retaining clearly invalid values as null.
8. Converted valid order dates to a consistent `yyyy-MM-dd` format and set invalid dates to null.
9. Saved the cleaned dataset in the Silver layer as CSV.
10. Read the Silver output back with Spark and verified the row count and schema.

In [18]:
spark.stop()